# coerce-float-arg-to-array — ex1: coerce_to_array: wrap int/float as 0-D tensor, pass-through others

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `coerce-float-arg-to-array`. Running the final beacon cell reports progress against the `Backprop: Coerce float arg to array` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Coerce float arg to array` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`coerce-float-arg-to-array`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "coerce-float-arg-to-array"
DD_SUBTOPIC = "Backprop: Coerce float arg to array"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Coerce float arg → tensor (boxed) — quick refresher

Some forward ops take a `float` constant: `multiply(t, 3.0)`, `add(t, 1.0)`, `pow(t, 2.0)`. The raw `torch` op accepts it fine (scalar broadcast), but our autograd wrapper must **coerce the float to a 0-D tensor** BEFORE running the forward so that:

1. The Recipe stores a `torch.Tensor` (not a Python float) at that argidx — downstream backward fns can do tensor math on it without a type-check.
2. The shape-of-output computation is consistent — `t.tensor(3.0)` broadcasts the same as the Python literal.

```python
def coerce_to_tensor(arg):
    if isinstance(arg, (int, float)):
        return t.tensor(float(arg))
    return arg
```

Note: the coerced value is a **leaf** with `requires_grad=False` — constants are not parents of the output, so they're skipped by `build_parents`. The coercion is purely about type uniformity for the forward call and Recipe storage.

### Exercise 1 — coerce_to_array: wrap int/float as 0-D tensor, pass-through others

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the float/int coercion step at the wrap_forward_fn entry: promote Python scalars to 0-D tensors so the Recipe and downstream back fns see uniform tensor types.
> Keywords: coerce, scalar-promotion, 0-d-tensor, wrap-forward
> ```

**KCs targeted:** `coerce-float-arg-to-array`, `unbox-args-tensor-to-array`

Implement `coerce_to_array(arg)`. The autograd wrapper sees calls like `multiply(t, 3.0)` where the second positional arg is a Python `float`. Before doing anything else, we promote it to a 0-D `torch.Tensor` so:

1. The Recipe stores a `torch.Tensor` (not a `float`) at that argidx — back fns can do tensor math on it without a type-check.
2. Forward-call broadcasting is consistent — `t.tensor(3.0)` broadcasts the same as the Python literal.

Rules:

- **`int` or `float` → `torch.tensor(float(arg))`.** Always promote to `float32` (the default for `torch.tensor(0.0)`). Even for `int` input — most downstream ops want float math anyway (`multiply`, `divide`, `pow`).
- **`torch.Tensor` → pass-through.** No copy. The whole point is that already-tensor args are left alone.
- **Anything else (tuple, str, None, ...) → pass-through.** Shape-tuples for `reshape` / `view` are legitimate non-tensor args and must not be wrapped.

Signature: `coerce_to_array(arg) -> any`. Result: tensor for scalars, original for everything else.

Note: do NOT use `isinstance(arg, (int, float, bool))` — bool is a subclass of int in Python, and we DON'T want to coerce `True`/`False` to a tensor (they're metadata flags like `keepdim`). Stick to `(int, float)` and explicitly check NOT `bool`.

In [ ]:
def coerce_to_array(arg):
    """Promote int/float to 0-D tensor; pass through tensors and other types."""
    raise NotImplementedError()


def _test_ex1():
    # --- float coerced to 0-D float tensor ---
    out = coerce_to_array(3.0)
    assert isinstance(out, t.Tensor), f'float must coerce to Tensor, got {type(out)}'
    assert out.ndim == 0, f'must be 0-D (scalar), got ndim={out.ndim}'
    assert out.dtype == t.float32, f'must be float32, got {out.dtype}'
    assert out.item() == 3.0

    # --- int coerced to 0-D float tensor (NOT int tensor) ---
    out = coerce_to_array(5)
    assert isinstance(out, t.Tensor)
    assert out.ndim == 0
    assert out.dtype == t.float32, (
        f'int must coerce to float (not int) — downstream ops want float math, '
        f'got {out.dtype}'
    )
    assert out.item() == 5.0

    # --- existing tensor passes through unchanged (identity) ---
    raw = t.tensor([1.0, 2.0, 3.0])
    out = coerce_to_array(raw)
    assert out is raw, 'tensor input must pass through (no copy)'

    # --- shape tuple (e.g. reshape arg) passes through ---
    out = coerce_to_array((3, 4))
    assert out == (3, 4)
    assert not isinstance(out, t.Tensor)

    # --- string / None / list pass through ---
    assert coerce_to_array('x') == 'x'
    assert coerce_to_array(None) is None
    assert coerce_to_array([1, 2]) == [1, 2]

    # --- bool must NOT be coerced (subclass-of-int trap) ---
    # True/False are commonly used as keepdim flags etc. — must pass through.
    out = coerce_to_array(True)
    assert out is True, (
        'bool must pass through (NOT be coerced) — '
        'bool is a subclass of int in Python; avoid (int, float) catching it'
    )
    assert coerce_to_array(False) is False

    # --- the coerced tensor broadcasts the same way the float would ---
    # this is the operational reason for coercion: forward call semantics unchanged.
    raw = t.tensor([1.0, 2.0, 3.0])
    scalar = coerce_to_array(2.0)
    assert t.allclose(raw * scalar, t.tensor([2.0, 4.0, 6.0])), (
        'coerced scalar must broadcast like the float literal'
    )

    # --- negative float ---
    out = coerce_to_array(-1.5)
    assert out.item() == -1.5
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def coerce_to_array(arg):
    # bool is a subclass of int — must check for it FIRST and pass through,
    # otherwise the (int, float) branch below would coerce True/False to tensors.
    if isinstance(arg, bool):
        return arg
    if isinstance(arg, (int, float)):
        return t.tensor(float(arg))
    return arg
```

**Why coerce at the wrapper entry, not in each back fn.** Doing it once in the wrapper means every downstream back fn can assume tensor args. Pushing the check into each back fn would mean N copies of the same scalar guard — and the Recipe would store heterogenous types, breaking introspection tools.

**Why promote `int` to `float` and not `int64`.** Most elementwise ops we'll backprop through (`multiply`, `divide`, `pow`, `add`) want floating-point math. `multiply_back0(grad, out, x, scalar)` does `grad * scalar` — if `scalar` is an int-tensor, PyTorch's type promotion can give surprising results. Keeping scalars as float32 sidesteps this.

**The bool trap.** `isinstance(True, int)` is `True` in Python — `bool` is literally a subclass of `int`. If you write `isinstance(arg, (int, float))` without an earlier bool guard, `coerce_to_array(True)` returns `tensor(1.0)` — but the caller passed `True` as a `keepdim` kwarg, not a numeric scalar. Now `sum(x, keepdim=tensor(1.0))` raises a confusing TypeError far from the cause.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()